# Foro PLN: decisiones de preprocesamiento para clasificar reseñas de hoteles en español

**Tarea:** clasificación binaria de sentimiento (negativa / positiva) de reseñas de hoteles.
**Dominio:** reseñas de TripAdvisor sobre hoteles de Andalucía (español peninsular, registro informal).
**Dataset:** *Andalusian Hotels' Reviews* (Chizhikova, Kaggle), versión balanceada `Balanced_AHR.csv`.
Licencia: CC BY-NC 4.0 (uso no comercial, con atribución). Incluye texto del corpus COAH (SINAI, 2014).

**Tramos**
1. Configuración y reproducibilidad
2. Auditoría de datos (duplicados, neutrales, fugas)
3. Unicode, acentos y mayúsculas
4. Emojis, hashtags, URLs y números
5. Stopwords, stemming y lematización
6. Tokenización y vocabulario
7. Baseline TF-IDF, eficiencia y sesgos

## 1. Configuración y reproducibilidad
Fijamos la semilla, registramos versiones y calculamos el hash SHA-256 del archivo de datos.
La carpeta no es un repositorio git, así que el *commit hash* queda como "no disponible"
y el hash de los datos cumple la función de identificar la entrada exacta.

In [1]:
import hashlib
import json
import logging
import platform
import random
import re
import subprocess
import sys
import unicodedata
from collections import Counter, defaultdict
from pathlib import Path

import emoji
import numpy as np
import pandas as pd
import sklearn
import spacy
from sklearn.model_selection import train_test_split

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
log = logging.getLogger("foro")

SEED: int = 42
random.seed(SEED)
np.random.seed(SEED)

DATA_PATH = Path("../../data/Balanced_AHR.csv")


def sha256_file(path: Path) -> str:
    """Calcula el SHA-256 de un archivo.

    Args:
        path: Ruta del archivo a resumir.

    Returns:
        Hash SHA-256 en hexadecimal.
    """
    digest = hashlib.sha256()
    with path.open("rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def git_commit() -> str:
    """Devuelve el hash del commit actual o un aviso si no hay repositorio.

    Returns:
        Hash del commit o un texto explicando que no está disponible.
    """
    try:
        out = subprocess.run(
            ["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=True
        )
        return out.stdout.strip()
    except (subprocess.CalledProcessError, FileNotFoundError):
        return "no-disponible (carpeta sin repositorio git)"


run_meta = {
    "seed": SEED,
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "scikit-learn": sklearn.__version__,
    "spacy": spacy.__version__,
    "emoji": emoji.__version__,
    "unicode_data_python": unicodedata.unidata_version,
    "data_file": DATA_PATH.name,
    "data_sha256": sha256_file(DATA_PATH),
    "data_bytes": DATA_PATH.stat().st_size,
    "commit": git_commit(),
}
print(json.dumps(run_meta, indent=2, ensure_ascii=False))

{
  "seed": 42,
  "python": "3.14.4",
  "platform": "Windows-11-10.0.26300-SP0",
  "pandas": "3.1.1",
  "numpy": "2.5.4",
  "scikit-learn": "1.9.1",
  "spacy": "3.8.18",
  "emoji": "2.16.0",
  "unicode_data_python": "16.0.0",
  "data_file": "Balanced_AHR.csv",
  "data_sha256": "6d3abc198db111d2a40d738a97070535d3afc02a7d16a7e55cbe637d610529be",
  "data_bytes": 4559782,
  "commit": "no-disponible (carpeta sin repositorio git)"
}


## 2. Auditoría de datos
Antes de preprocesar texto hay que saber **qué** estamos preprocesando.
Revisamos estructura, la etiqueta neutral (`3`) y el texto duplicado.

In [2]:
raw = pd.read_csv(DATA_PATH, encoding="utf-8").drop(columns=["Unnamed: 0"])
print("forma:", raw.shape)
print()
print(raw.dtypes.to_string())
print()
print("nulos por columna:")
print(raw.isna().sum().to_string())
print()
print("rating (filas) vs label (columnas):")
print(pd.crosstab(raw["rating"], raw["label"]).to_string())

forma: (7615, 6)

title            str
rating         int64
review_text      str
location         str
hotel            str
label          int64

nulos por columna:
title            0
rating           0
review_text      0
location       796
hotel          796
label            0

rating (filas) vs label (columnas):
label      0     1     3
rating                  
1       1677     0     0
2        994     0     0
3          0     0  2274
5          0  2670     0


La tabla de arriba aclara **qué significa cada etiqueta** a partir de los datos y no solo de la
documentación. La etiqueta `3` agrupa las reseñas neutrales.

In [3]:
dup_any = raw.duplicated("review_text", keep=False)
dup_extra = raw.duplicated("review_text", keep="first")
sizes = raw[dup_any].groupby("review_text").size()

print(f"filas totales:                         {len(raw)}")
print(f"filas que comparten texto con otra:    {int(dup_any.sum())}")
print(f"filas sobrantes (copias tras la 1ª):   {int(dup_extra.sum())}")
print(f"textos distintos repetidos:            {len(sizes)}")
print()
print("tamaño del grupo repetido -> cuántos grupos:")
print(sizes.value_counts().sort_index().to_string())

label_conflict = int((raw[dup_any].groupby("review_text")["label"].nunique() > 1).sum())
print(f"\ngrupos con etiquetas contradictorias:  {label_conflict}")

full_dup = int(raw.duplicated(["title", "review_text", "hotel", "label"], keep="first").sum())
print(f"copias idénticas en título+texto+hotel+label: {full_dup}")

print(f"\n% de hotel nulo, todo el dataset:        {raw['hotel'].isna().mean():.1%}")
print(f"% de hotel nulo, filas en grupos repetidos: {raw.loc[dup_any, 'hotel'].isna().mean():.1%}")

filas totales:                         7615
filas que comparten texto con otra:    4600
filas sobrantes (copias tras la 1ª):   2514
textos distintos repetidos:            2086

tamaño del grupo repetido -> cuántos grupos:
2     1858
3       81
4      130
5        1
6       12
8        1
12       3

grupos con etiquetas contradictorias:  0
copias idénticas en título+texto+hotel+label: 2514

% de hotel nulo, todo el dataset:        10.5%
% de hotel nulo, filas en grupos repetidos: 0.5%


### Decisión de limpieza (valores por defecto, a confirmar)
`KEEP_NEUTRAL` y `DEDUP` son parámetros: cambia su valor y vuelve a ejecutar para ver el efecto.

In [4]:
KEEP_NEUTRAL: bool = False   # False: clasificación binaria, se descartan las de 3 estrellas
DEDUP: bool = True           # True: se conserva una sola copia de cada texto

df = raw.copy()
if not KEEP_NEUTRAL:
    df = df[df["label"] != 3]
if DEDUP:
    df = df.drop_duplicates("review_text", keep="first")
df = df.reset_index(drop=True)

print(f"antes: {len(raw)} filas  ->  después: {len(df)} filas")
print("\nbalance de clases después de limpiar:")
print(df["label"].value_counts().sort_index().to_string())
print((df["label"].value_counts(normalize=True).sort_index() * 100).round(1).to_string())

antes: 7615 filas  ->  después: 3361 filas

balance de clases después de limpiar:
label
0    2053
1    1308
label
0    61.1
1    38.9


### Evidencia de fuga de datos (*data leakage*)
Medimos cuántos textos del conjunto de prueba aparecen también en entrenamiento
según si se eliminaron o no los duplicados antes de dividir.

In [5]:
def leak_rate(frame: pd.DataFrame, seed: int = SEED) -> float:
    """Fracción de textos de prueba que también están en entrenamiento.

    Args:
        frame: Tabla con columnas ``review_text`` y ``label``.
        seed: Semilla de la partición.

    Returns:
        Proporción en [0, 1] de textos de prueba presentes en entrenamiento.
    """
    train, test = train_test_split(
        frame, test_size=0.2, stratify=frame["label"], random_state=seed
    )
    return float(test["review_text"].isin(set(train["review_text"])).mean())


binary_with_dups = raw[raw["label"] != 3].reset_index(drop=True)
print(f"con duplicados ({len(binary_with_dups)} filas): {leak_rate(binary_with_dups):.1%} del test ya está en train")
print(f"sin duplicados ({len(df)} filas):      {leak_rate(df):.1%} del test ya está en train")

con duplicados (5341 filas): 57.0% del test ya está en train
sin duplicados (3361 filas):      0.0% del test ya está en train


### Tu decisión (tramo 2)
Responde con tus palabras antes de seguir:

1. ¿Por qué quitar duplicados **antes** de dividir en entrenamiento y prueba, y no después?
2. ¿Qué pasa con el significado de "clasificación binaria" si descartas las reseñas de 3 estrellas?
   ¿Qué le dirías a un lector del foro sobre esa limitación?
3. Mi decisión sobre duplicados y neutrales: **________**

## 3. Unicode, acentos y mayúsculas
### 3.1 Codificación: ¿hay caracteres sospechosos?
El archivo se leyó como UTF-8 sin errores. Aun así buscamos señales de problemas típicos:
carácter de reemplazo, espacios invisibles y *mojibake* (texto UTF-8 mal decodificado como Latin-1).

In [6]:
texts = df["review_text"]
joined = "".join(texts)
non_ascii = Counter(c for c in joined if ord(c) > 127)

print(f"caracteres totales: {len(joined):,}")
print(f"caracteres no ASCII: {sum(non_ascii.values()):,} ({sum(non_ascii.values()) / len(joined):.2%})")
print(f"caracteres no ASCII distintos: {len(non_ascii)}\n")

print("los 25 más frecuentes fuera de ASCII:")
for ch, n in non_ascii.most_common(25):
    print(f"  {ch!r:>6}  U+{ord(ch):04X}  {unicodedata.name(ch, '?'):<38} {n:>7,}")

SUSPECT = {
    "U+FFFD carácter de reemplazo": "�",
    "U+00A0 espacio de no separación": " ",
    "U+200B espacio de ancho cero": "​",
    "U+FEFF marca BOM": "﻿",
    "U+00AD guion blando": "­",
}
print("\ntextos que contienen caracteres sospechosos:")
for name, ch in SUSPECT.items():
    print(f"  {name:<34} {int(texts.str.contains(ch, regex=False).sum()):>6}")

mojibake = int(texts.str.contains(r"Ã[\u0080-¿]|Â[ -¿]", regex=True).sum())
print(f"  {'patrones de mojibake (Ã©, Ã±, Â...)':<34} {mojibake:>6}")

ctrl = Counter(repr(c) for c in joined if unicodedata.category(c) == "Cc")
print(f"\ncaracteres de control presentes: {dict(ctrl)}")

caracteres totales: 1,766,156
caracteres no ASCII: 26,975 (1.53%)
caracteres no ASCII distintos: 97

los 25 más frecuentes fuera de ASCII:
     'ó'  U+00F3  LATIN SMALL LETTER O WITH ACUTE          8,018
     'í'  U+00ED  LATIN SMALL LETTER I WITH ACUTE          6,431
     'á'  U+00E1  LATIN SMALL LETTER A WITH ACUTE          3,925
     'ñ'  U+00F1  LATIN SMALL LETTER N WITH TILDE          3,053
     'é'  U+00E9  LATIN SMALL LETTER E WITH ACUTE          2,655
     'ú'  U+00FA  LATIN SMALL LETTER U WITH ACUTE          1,190
     '…'  U+2026  HORIZONTAL ELLIPSIS                        583
     '€'  U+20AC  EURO SIGN                                  230
     '¡'  U+00A1  INVERTED EXCLAMATION MARK                  143
     '¿'  U+00BF  INVERTED QUESTION MARK                      64
     '“'  U+201C  LEFT DOUBLE QUOTATION MARK                  62
     '”'  U+201D  RIGHT DOUBLE QUOTATION MARK                 61
     'ü'  U+00FC  LATIN SMALL LETTER U WITH DIAERESIS         58
     'º'  U+00BA


caracteres de control presentes: {"'\\n'": 500}


### 3.2 Formas de normalización: NFC, NFD, NFKC, NFKD
Como lo vimos en los notebooks de ejemplo un mismo "é" puede ser un solo carácter (U+00E9) o dos (e + acento combinado U+0301).
Si el texto mezcla ambas representaciones, el modelo ve palabras distintas donde un humano ve una.

In [7]:
rows = []
for form in ["NFC", "NFD", "NFKC", "NFKD"]:
    normalized = texts.map(lambda s, f=form: unicodedata.normalize(f, s))
    rows.append(
        {
            "forma": form,
            "textos_que_cambian": int((normalized != texts).sum()),
            "pct_textos": round(float((normalized != texts).mean() * 100), 2),
            "caracteres_distintos": len(set("".join(normalized))),
            "largo_total": int(normalized.str.len().sum()),
        }
    )
display(pd.DataFrame(rows))

changed_by_nfkc = [
    (c, unicodedata.normalize("NFKC", c)) for c in non_ascii if unicodedata.normalize("NFKC", c) != c
]
print("\ncaracteres que NFKC transformaría:")
for src, dst in changed_by_nfkc:
    print(f"  {src!r} U+{ord(src):04X} {unicodedata.name(src, '?')}  ->  {dst!r}")
if not changed_by_nfkc:
    print("  (ninguno)")

,forma,textos_que_cambian,pct_textos,caracteres_distintos,largo_total
0,NFC,0,0.00,185,1766156
1,NFD,3180,94.61,163,1791744
2,NFKC,571,16.99,180,1767338
3,NFKD,3188,94.85,156,1792926



caracteres que NFKC transformaría:
  '…' U+2026 HORIZONTAL ELLIPSIS  ->  '...'
  'º' U+00BA MASCULINE ORDINAL INDICATOR  ->  'o'
  'ª' U+00AA FEMININE ORDINAL INDICATOR  ->  'a'
  '¨' U+00A8 DIAERESIS  ->  ' ̈'
  '\xa0' U+00A0 NO-BREAK SPACE  ->  ' '
  '²' U+00B2 SUPERSCRIPT TWO  ->  '2'
  '´' U+00B4 ACUTE ACCENT  ->  ' ́'


### 3.3 Acentos: ¿conviene quitarlos?
Quitar acentos reduce el vocabulario, pero puede **fusionar palabras distintas**
(`esta`/`está`, `si`/`sí`, `mas`/`más`). Buscamos colisiones reales en este corpus.

In [8]:
WORD = re.compile(r"\w+", re.UNICODE)


def strip_accents(s: str) -> str:
    """Elimina marcas diacríticas combinadas tras descomponer en NFD.

    Nota: esto también convierte ``ñ`` en ``n``.

    Args:
        s: Cadena de entrada.

    Returns:
        Cadena sin diacríticos.
    """
    return "".join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn")


vocab = Counter(w.lower() for t in texts for w in WORD.findall(t))
buckets: dict[str, list[tuple[str, int]]] = defaultdict(list)
for word, n in vocab.items():
    buckets[strip_accents(word)].append((word, n))

collisions = {k: sorted(v, key=lambda x: -x[1]) for k, v in buckets.items() if len(v) > 1}
frequent = {k: v for k, v in collisions.items() if v[1][1] >= 20}

print(f"vocabulario (minúsculas, con acentos):  {len(vocab):,}")
print(f"vocabulario (minúsculas, sin acentos):  {len(buckets):,}")
print(f"formas que se fusionan al quitar acentos: {len(collisions):,}")
print(f"  de ellas, con ambas variantes >= 20 apariciones: {len(frequent)}\n")

print("colisiones más frecuentes (ambas variantes comunes):")
top = sorted(frequent.items(), key=lambda kv: -sum(n for _, n in kv[1]))[:15]
for key, variants in top:
    print(f"  {key:<12} <- " + " | ".join(f"{w} ({n:,})" for w, n in variants))

vocabulario (minúsculas, con acentos):  18,478
vocabulario (minúsculas, sin acentos):  17,393
formas que se fusionan al quitar acentos: 1,039
  de ellas, con ambas variantes >= 20 apariciones: 56

colisiones más frecuentes (ambas variantes comunes):
  que          <- que (10,257) | qué (84) | què (1)
  el           <- el (8,750) | él (53)
  se           <- se (2,613) | sé (42)
  habitacion   <- habitación (2,049) | habitacion (273) | habitaciòn (4) | habitacíón (2) | habítacion (1)
  mas          <- más (955) | mas (389)
  si           <- si (1,213) | sí (101) | sì (1)
  esta         <- está (692) | esta (568) | ésta (12)
  como         <- como (1,053) | cómo (51) | còmo (1)
  este         <- este (1,039) | éste (32) | esté (29)
  mi           <- mi (925) | mí (26) | mì (1)
  recepcion    <- recepción (701) | recepcion (70) | recepciòn (3)
  habia        <- había (551) | habia (152) | habìa (1)
  solo         <- solo (500) | sólo (180)
  dia          <- día (465) | dia (174) | dìa (3)


### 3.4 Mayúsculas
Pasar a minúsculas reduce vocabulario, pero se pierde el énfasis (`PÉSIMO`, `NO RECOMIENDO`).
Medimos si las mayúsculas sostenidas aportan señal de sentimiento en este corpus.

In [9]:
def caps_ratio(s: str) -> float:
    """Proporción de letras mayúsculas respecto a todas las letras.

    Args:
        s: Cadena de entrada.

    Returns:
        Valor en [0, 1]; 0.0 si no hay letras.
    """
    letters = [c for c in s if c.isalpha()]
    return sum(c.isupper() for c in letters) / len(letters) if letters else 0.0


df["caps_ratio"] = df["review_text"].map(caps_ratio)
df["title_caps_ratio"] = df["title"].fillna("").map(caps_ratio)

summary = df.groupby("label").agg(
    n=("label", "size"),
    caps_media=("caps_ratio", "mean"),
    pct_texto_con_mas_30pct_mayus=("caps_ratio", lambda x: (x > 0.30).mean() * 100),
    pct_titulo_con_mas_50pct_mayus=("title_caps_ratio", lambda x: (x > 0.50).mean() * 100),
)
display(summary.round(2))

vocab_cased = {w for t in texts for w in WORD.findall(t)}
print(f"\nvocabulario respetando mayúsculas: {len(vocab_cased):,}")
print(f"vocabulario en minúsculas:         {len(vocab):,}")

,n,caps_media,pct_texto_con_mas_30pct_mayus,pct_titulo_con_mas_50pct_mayus
label,,,,
0,2053,0.02,0.24,9.94
1,1308,0.02,0.00,6.80



vocabulario respetando mayúsculas: 20,988
vocabulario en minúsculas:         18,478


### Tu decisión (tramo 3)
1. **Forma Unicode:** ¿NFC o NFKC? Justifica con lo que viste en la tabla 3.2.
2. **Acentos:** ¿los conservas o los quitas? Usa una colisión concreta de 3.3 como ejemplo.
3. **Mayúsculas:** ¿`lower()` siempre, o conservas alguna señal de énfasis? ¿Qué muestra 3.4?
4. Mi decisión: **________**

*Pista de revisión crítica:* una conclusión sobre "las mayúsculas indican enojo" necesita más
evidencia que una diferencia de promedios. ¿Qué otra comprobación harías?

## 4. Emojis, hashtags, menciones, URLs y números
Primero **medimos qué hay** en este corpus. Una regla de limpieza que no aplica a tus datos
(por ejemplo, hashtags en reseñas de hotel) solo añade complejidad.

In [10]:
PATTERNS: dict[str, str | None] = {
    "emoji": None,  # se cuenta con la librería emoji
    "hashtag": r"#\w+",
    "arroba (mención o @s inclusivo)": r"@\w+",
    "url": r"https?://\S+|www\.\S+",
    "correo": r"[\w.+-]+@[\w-]+\.[\w.]+",
    "precio": r"\d+(?:[.,]\d+)?\s?(?:euros?\b|€)|€\s?\d+(?:[.,]\d+)?",
    "estrellas del hotel": r"\d\s?(?:\*|estrellas?)",
    "hora": r"\b\d{1,2}[:h]\d{2}\b",
    "ordinal o nº": r"\d+\s?[ºª]|\bn[ºo°]\s?\d+",
    "cualquier número": r"\d+(?:[.,]\d+)?",
}

rows = []
for name, rx in PATTERNS.items():
    if rx is None:
        found = texts.map(lambda s: [e["emoji"] for e in emoji.emoji_list(s)])
    else:
        found = texts.str.findall(rx, flags=re.IGNORECASE)
    rows.append(
        {
            "elemento": name,
            "textos": int((found.map(len) > 0).sum()),
            "pct_textos": round(float((found.map(len) > 0).mean() * 100), 2),
            "apariciones": int(found.map(len).sum()),
            "ejemplos": [x for lst in found for x in lst][:4],
        }
    )
display(pd.DataFrame(rows))

,elemento,textos,pct_textos,apariciones,ejemplos
0,emoji,34,1.01,88,"[😳, 😭, 😥, 👍]"
1,hashtag,0,0.00,0,[]
2,arroba (mención o @s inclusivo),9,0.27,14,"[@s, @s, @s, @s]"
3,url,1,0.03,1,[http://www.]
4,correo,0,0.00,0,[]
5,precio,321,9.55,405,"[10 euros, 70 euros, 5€, 10 euros]"
6,estrellas del hotel,241,7.17,300,"[4 estrellas, 4*, 4*, 4 estrellas]"
7,hora,91,2.71,130,"[15:00, 22:15, 22:18, 1:30]"
8,ordinal o nº,39,1.16,59,"[360º, 1ª, 1ª, nº 4]"
9,cualquier número,1625,48.35,3865,"[6, 7, 6, 1]"


### 4.1 Emojis: ¿aportan señal?

In [11]:
has_emoji = texts.map(lambda s: emoji.emoji_count(s) > 0)
print("textos con emoji por etiqueta (0=negativa, 1=positiva):")
print(pd.crosstab(df["label"], has_emoji).to_string())

for lbl in (0, 1):
    common = Counter(
        e["emoji"] for s in texts[df["label"] == lbl] for e in emoji.emoji_list(s)
    ).most_common(8)
    print(f"\nemojis más comunes, etiqueta {lbl}: {common}")

print("\ndemojize en español:")
for e in ["🤬", "👎", "🌟", "😍"]:
    print(f"  {e} -> {emoji.demojize(e, language='es')}")

textos con emoji por etiqueta (0=negativa, 1=positiva):
review_text  False  True 
label                    
0             2035     18
1             1292     16



emojis más comunes, etiqueta 0: [('👎', 6), ('😂', 5), ('🤬', 5), ('💩', 5), ('🤫', 3), ('😡', 2), ('😁', 2), ('👏🏼', 2)]



emojis más comunes, etiqueta 1: [('🌟', 7), ('🤗', 4), ('😘', 4), ('😊', 3), ('😍', 3), ('🥰', 3), ('♥️', 3), ('❤️', 2)]

demojize en español:
  🤬 -> :cara_con_símbolos_en_la_boca:
  👎 -> :pulgar_hacia_abajo:
  🌟 -> :estrella_brillante:
  😍 -> :cara_sonriendo_con_ojos_de_corazón:


### 4.2 Arrobas: ¿menciones o español inclusivo?
En reseñas de hotel no esperamos menciones a usuarios. Miramos los contextos reales.

In [12]:
ctx = texts.str.extractall(r"(.{0,18}@\w+.{0,8})")[0]
print(f"apariciones de @palabra: {len(ctx)}")
for c in ctx.head(8):
    print(" ", repr(c))

apariciones de @palabra: 13
  'er, siendo objetiv@s la pens'
  'e atento y muy maj@s con l@s'
  '  Eso sí, los chic@s de la r'
  'n aparecido azafat@s con el '
  'SEÑORES@s Bueno q'
  'Hola viajer@s! Buscar'
  'que tampoco vosotr@s tengáis'
  '. Un saludo viajer@s!!'


### 4.3 Números: ¿ruido o señal?

In [13]:
has_digit = texts.str.contains(r"\d")
print("proporción de textos con algún dígito, por etiqueta:")
print(pd.crosstab(df["label"], has_digit, normalize="index").round(3).to_string())

price = texts.str.contains(PATTERNS["precio"], flags=re.IGNORECASE, regex=True)
hour = texts.str.contains(PATTERNS["hora"], regex=True)
print("\ntextos con precio, por etiqueta:   ", df.groupby("label").apply(lambda g: price[g.index].mean()).round(3).to_dict())
print("textos con hora, por etiqueta:     ", df.groupby("label").apply(lambda g: hour[g.index].mean()).round(3).to_dict())

proporción de textos con algún dígito, por etiqueta:
review_text  False  True 
label                    
0            0.419  0.581
1            0.670  0.330

textos con precio, por etiqueta:    {0: 0.143, 1: 0.021}
textos con hora, por etiqueta:      {0: 0.043, 1: 0.002}


### 4.4 Función de limpieza parametrizada (propuesta inicial)
Cada decisión es un parámetro, de modo que luego podamos **comparar** variantes con evidencia.
Reglas actuales:
- URL → `url` · hashtag → se quita `#` y se conserva la palabra · `@s` inclusivo → se deja intacto
- precio / hora / estrellas / ordinal → marcadores tipados (`num_precio`, `num_hora`, `num_estrellas`, `num_ordinal`); otro número → `num`
- emoji → se traduce a su nombre en español (`emoji_...`) o se elimina (`keep_emoji=False`)

In [14]:
URL_RX = re.compile(r"https?://\S*|www\.\S+", re.IGNORECASE)
HASHTAG_RX = re.compile(r"#(\w+)")
STARS_RX = re.compile(r"\d\s?(?:\*|estrellas?)", re.IGNORECASE)
PRICE_RX = re.compile(r"\d+(?:[.,]\d+)?\s?(?:euros?\b|€)|€\s?\d+(?:[.,]\d+)?", re.IGNORECASE)
TIME_RX = re.compile(r"\b\d{1,2}[:h]\d{2}\b", re.IGNORECASE)
ORDINAL_RX = re.compile(r"\d+\s?[ºª]|\bn[ºo°]\s?\d+", re.IGNORECASE)
NUMBER_RX = re.compile(r"\d+(?:[.,]\d+)?")
SPACES_RX = re.compile(r"\s+")


def clean_text(
    s: str,
    *,
    form: str = "NFC",
    lower: bool = True,
    strip_acc: bool = False,
    keep_emoji: bool = True,
) -> str:
    """Aplica la limpieza propuesta a una reseña.

    Args:
        s: Texto original.
        form: Forma de normalización Unicode (``NFC``, ``NFKC``...).
        lower: Si es True, convierte a minúsculas.
        strip_acc: Si es True, elimina diacríticos (también convierte ``ñ`` en ``n``).
        keep_emoji: Si es True, traduce emojis a ``emoji_<nombre>``; si no, los elimina.

    Returns:
        Texto limpio, con espacios colapsados.
    """
    s = unicodedata.normalize(form, s)
    s = URL_RX.sub(" url ", s)
    s = HASHTAG_RX.sub(r" \1 ", s)
    s = STARS_RX.sub(" num_estrellas ", s)
    s = PRICE_RX.sub(" num_precio ", s)
    s = TIME_RX.sub(" num_hora ", s)
    s = ORDINAL_RX.sub(" num_ordinal ", s)
    s = NUMBER_RX.sub(" num ", s)
    if keep_emoji:
        s = emoji.demojize(s, language="es", delimiters=(" emoji_", " "))
    else:
        s = emoji.replace_emoji(s, replace=" ")
    if lower:
        s = s.lower()
    if strip_acc:
        s = strip_accents(s)
    return SPACES_RX.sub(" ", s).strip()


def shortest_match(rx: str, min_len: int = 60) -> str:
    """Devuelve el texto más corto (>= min_len) que contiene el patrón.

    Args:
        rx: Expresión regular a buscar.
        min_len: Largo mínimo del texto para evitar ejemplos triviales.

    Returns:
        El texto original más corto que cumple la condición.
    """
    sub = texts[texts.str.contains(rx, regex=True, flags=re.IGNORECASE)]
    sub = sub[sub.str.len() >= min_len]
    return sub.loc[sub.str.len().idxmin()]


print("=== ANTES / DESPUÉS (evidencia para el post) ===")
for label, rx in {
    "emoji": r"[\U0001F300-\U0001FAFF]",
    "precio": r"\d\s?euros|€",
    "estrellas": r"\d\s?(?:\*|estrellas)",
    "hora": r"\b\d{1,2}[:h]\d{2}\b",
    "url": r"https?://",
}.items():
    original = shortest_match(rx)
    m = re.search(rx, original, flags=re.IGNORECASE)
    snippet = original[max(0, m.start() - 110) : m.end() + 110]
    print(f"\n[{label}]")
    print("  antes:  ", snippet)
    print("  después:", clean_text(snippet))

=== ANTES / DESPUÉS (evidencia para el post) ===

[emoji]
  antes:    Tomar café en sus patios es algo sencillamente maravilloso. Sin duda recomendable al 100%, experiencia única 🥰🥰
  después: tomar café en sus patios es algo sencillamente maravilloso. sin duda recomendable al num %, experiencia única emoji_cara_sonriendo_con_corazones emoji_cara_sonriendo_con_corazones

[precio]
  antes:   Menú de 15€, en el q tanto los, primeros, los segundos y los postres, dejan bastante q desear, y más aún teniendo en cuen
  después: menú de num_precio , en el q tanto los, primeros, los segundos y los postres, dejan bastante q desear, y más aún teniendo en cuen



[estrellas]
  antes:   Situado en el Arenal se encuentra este hotel de 5 estrellas que está decorado con un gusto exquisito . Hemos recibido  una atención del personal maravillosa. Es un hotel
  después: situado en el arenal se encuentra este hotel de num_estrellas que está decorado con un gusto exquisito . hemos recibido una atención del personal maravillosa. es un hotel

[hora]
  antes:   Limpio y muy bien ubicado para conocer Sevilla. La única pega quizás es que la hora de entrada es a las 16:00 y la salida a las 11:00, pero aún así puedes dejar las maletas en recepción desde las 9:00
  después: limpio y muy bien ubicado para conocer sevilla. la única pega quizás es que la hora de entrada es a las num_hora y la salida a las num_hora , pero aún así puedes dejar las maletas en recepción desde las num_hora

[url]
  antes:   son, de los antiguos y sin TDT, o sea, no hay televisores.(Igual hay que pedir el mando y el TDT en recepción http://www. hotelcasablancagranada. com/es/hotel-gran

### Tu decisión (tramo 4)
1. **Emojis:** ¿los conservas (traducidos a texto), los eliminas o los dejas como están? Usa lo que viste en 4.1.
2. **Números:** ¿los borras, los conservas o los conviertes en marcadores tipados? ¿Qué muestra 4.3?
3. **Hashtags y URLs:** ¿por qué esta regla casi no afecta a este corpus? ¿Qué cambiaría en un corpus de tweets?
4. **`@s`:** ¿por qué sería un error tratarlo como mención de usuario?
5. Mi decisión: **________**

## 5. Stopwords, stemming y lematización
### 5.1 ¿Qué se pierde al quitar stopwords?
Las listas genéricas incluyen **negaciones e intensificadores** (`no`, `nunca`, `sin`, `muy`, `pero`, `bien`, `mal`).
En sentimiento eso es peligroso: "no me gustó" sin `no` significa lo contrario.

In [15]:
import os
import time

import psutil
from nltk.stem.snowball import SpanishStemmer
from spacy.lang.es.stop_words import STOP_WORDS

proc = psutil.Process(os.getpid())


def rss_mb() -> float:
    """Memoria residente del proceso actual.

    Returns:
        Memoria en megabytes.
    """
    return proc.memory_info().rss / 1e6


clean = texts.map(clean_text)
tok = clean.map(WORD.findall)

counts = {lbl: Counter(w for ts in tok[df["label"] == lbl] for w in ts) for lbl in (0, 1)}
total = {lbl: sum(c.values()) for lbl, c in counts.items()}

rows = []
for w in STOP_WORDS:
    n0, n1 = counts[0][w], counts[1][w]
    if n0 + n1 >= 50:
        rows.append(
            {
                "palabra": w,
                "neg_por_mil": 1000 * n0 / total[0],
                "pos_por_mil": 1000 * n1 / total[1],
                "razon_neg/pos": ((n0 + 1) / total[0]) / ((n1 + 1) / total[1]),
            }
        )
sw = pd.DataFrame(rows)
sw["_abs_log"] = np.abs(np.log(sw["razon_neg/pos"]))
print(f"stopwords en la lista de spaCy: {len(STOP_WORDS)}")
print("\nstopwords cuya frecuencia más difiere entre clases (n >= 50):")
display(sw.sort_values("_abs_log", ascending=False).head(15).drop(columns="_abs_log").round(2))

stopwords en la lista de spaCy: 521

stopwords cuya frecuencia más difiere entre clases (n >= 50):


,palabra,neg_por_mil,pos_por_mil,razon_neg/pos
238,peor,1.36,0.03,35.19
77,nadie,0.95,0.03,24.75
119,mal,1.89,0.09,18.36
108,dijo,0.45,0.01,17.65
245,tampoco,0.57,0.03,14.76
30,dicen,0.49,0.03,12.65
19,habia,0.64,0.05,9.92
223,ni,3.86,0.44,8.56
143,dijeron,0.54,0.05,8.39
147,ese,0.72,0.09,7.03


### 5.2 Lista curada
Partimos de la lista de spaCy y **devolvemos** al texto las palabras que cargan polaridad o intensidad.

In [16]:
KEEP = {
    "no", "nunca", "jamás", "ni", "sin", "tampoco", "nada", "nadie", "ningún", "ninguna", "ninguno",
    "muy", "más", "menos", "poco", "pocos", "pocas", "bastante", "demasiado", "demasiada", "tan", "tanto",
    "pero", "sino", "aunque", "bien", "mal", "mejor", "peor", "buen", "sí",
}
CURATED = STOP_WORDS - KEEP

print(f"devueltas al texto (estaban en la lista): {sorted(KEEP & STOP_WORDS)}")
print(f"no estaban en la lista de spaCy:          {sorted(KEEP - STOP_WORDS)}")
print(f"tamaño lista completa: {len(STOP_WORDS)}  |  lista curada: {len(CURATED)}")


def drop_words(tokens: list[str], stop: set[str]) -> list[str]:
    """Quita los tokens que están en ``stop``.

    Args:
        tokens: Lista de tokens.
        stop: Conjunto de palabras a eliminar.

    Returns:
        Lista filtrada.
    """
    return [t for t in tokens if t not in stop]


demo = "El hotel no está bien, pero la ubicación es muy buena y nunca esperé tanto"
demo_tokens = WORD.findall(clean_text(demo))
print("\nejemplo ilustrativo (frase inventada para mostrar el efecto):")
print("  original:        ", demo_tokens)
print("  lista completa:  ", drop_words(demo_tokens, STOP_WORDS))
print("  lista curada:    ", drop_words(demo_tokens, CURATED))

rows = []
for name, stop in [("sin quitar", set()), ("lista completa", STOP_WORDS), ("lista curada", CURATED)]:
    kept = tok.map(lambda ts, s=stop: drop_words(ts, s))
    rows.append(
        {
            "variante": name,
            "tokens_por_reseña": round(float(kept.map(len).mean()), 1),
            "vocabulario": len({w for ts in kept for w in ts}),
        }
    )
display(pd.DataFrame(rows))

devueltas al texto (estaban en la lista): ['aunque', 'bastante', 'bien', 'buen', 'demasiado', 'mal', 'mejor', 'menos', 'muy', 'más', 'nada', 'nadie', 'ni', 'ninguna', 'ninguno', 'ningún', 'no', 'nunca', 'peor', 'pero', 'pocas', 'poco', 'pocos', 'sin', 'sino', 'sí', 'tampoco', 'tan', 'tanto']
no estaban en la lista de spaCy:          ['demasiada', 'jamás']
tamaño lista completa: 521  |  lista curada: 492

ejemplo ilustrativo (frase inventada para mostrar el efecto):
  original:         ['el', 'hotel', 'no', 'está', 'bien', 'pero', 'la', 'ubicación', 'es', 'muy', 'buena', 'y', 'nunca', 'esperé', 'tanto']
  lista completa:   ['hotel', 'ubicación', 'esperé']
  lista curada:     ['hotel', 'no', 'bien', 'pero', 'ubicación', 'muy', 'nunca', 'esperé', 'tanto']


,variante,tokens_por_reseña,vocabulario
0,sin quitar,92.3,18168
1,lista completa,38.2,17677
2,lista curada,44.3,17706


### 5.3 Stemming (Snowball) frente a lematización (spaCy)
El *stemmer* corta sufijos con reglas y es muy rápido, pero produce raíces que no son palabras.
El lematizador de spaCy usa el contexto (etiqueta gramatical) y devuelve palabras reales, a costa de tiempo y memoria.
Medimos ambos sobre el mismo texto limpio.

In [17]:
stemmer = SpanishStemmer()
vocab_tokens = sorted({w for ts in tok for w in ts if w.isalpha()})

t0 = time.perf_counter()
stem_of = {w: stemmer.stem(w) for w in vocab_tokens}
t_stem = time.perf_counter() - t0

rss_before_load = rss_mb()
t0 = time.perf_counter()
nlp = spacy.load("es_core_news_sm", disable=["parser", "ner"])
t_load = time.perf_counter() - t0
rss_after_load = rss_mb()

pairs: Counter = Counter()
t0 = time.perf_counter()
for doc in nlp.pipe(clean.tolist(), batch_size=64):
    for t in doc:
        if t.is_alpha:
            pairs[(t.text, t.lemma_)] += 1
t_lemma = time.perf_counter() - t0
rss_after_run = rss_mb()

lemma_of: dict[str, str] = {}
for (word, lemma), _n in pairs.most_common():
    lemma_of.setdefault(word, lemma)

rows = [
    {"método": "sin reducir", "vocabulario": len(vocab_tokens), "segundos": 0.0},
    {"método": "stemming Snowball", "vocabulario": len(set(stem_of.values())), "segundos": round(t_stem, 3)},
    {"método": "lematización spaCy", "vocabulario": len(set(lemma_of.values())), "segundos": round(t_lemma, 1)},
]
table = pd.DataFrame(rows)
table["reducción_%"] = ((1 - table["vocabulario"] / len(vocab_tokens)) * 100).round(1)
display(table)
print(f"\ncarga del modelo spaCy: {t_load:.1f} s, memoria +{rss_after_load - rss_before_load:.0f} MB")
print(f"tras lematizar {len(clean)} reseñas: memoria total del proceso {rss_after_run:.0f} MB")

,método,vocabulario,segundos,reducción_%
0,sin reducir,18121,0.000,0.0
1,stemming Snowball,9009,0.105,50.3
2,lematización spaCy,12540,15.100,30.8



carga del modelo spaCy: 0.3 s, memoria +58 MB
tras lematizar 3361 reseñas: memoria total del proceso 388 MB


In [18]:
sample = [
    "habitaciones", "habitación", "limpias", "limpio", "limpieza", "recomendable", "recomiendo",
    "recepcionista", "horrible", "gustó", "gustaron", "atendieron", "atención", "trato", "tratan",
]
present = [w for w in sample if w in stem_of]
display(
    pd.DataFrame(
        {
            "palabra": present,
            "stem (Snowball)": [stem_of[w] for w in present],
            "lema (spaCy)": [lemma_of.get(w, "—") for w in present],
        }
    )
)

groups: dict[str, list[str]] = defaultdict(list)
for w, s_ in stem_of.items():
    groups[s_].append(w)
big = sorted(groups.items(), key=lambda kv: -len(kv[1]))[:8]
print("\nraíces que agrupan más palabras distintas (revisa si las fusiones tienen sentido):")
for s_, ws in big:
    print(f"  {s_:<10} <- {', '.join(ws[:10])}{' ...' if len(ws) > 10 else ''}  ({len(ws)} formas)")

,palabra,stem (Snowball),lema (spaCy)
0,habitaciones,habit,habitación
1,habitación,habit,habitación
2,limpias,limpi,limpio
3,limpio,limpi,limpio
4,limpieza,limpiez,limpieza
5,recomendable,recomend,recomendable
6,recomiendo,recom,recomeir
7,recepcionista,recepcion,recepcionista
8,horrible,horribl,horrible
9,gustó,gust,gustar



raíces que agrupan más palabras distintas (revisa si las fusiones tienen sentido):
  pas        <- pas, pasa, pasaba, pasaban, pasabas, pasad, pasada, pasadas, pasado, pasados ...  (40 formas)
  dej        <- dej, deja, dejaba, dejaban, dejada, dejadas, dejado, dejamos, dejan, dejando ...  (39 formas)
  llam       <- llama, llamaba, llamabas, llamad, llamada, llamadas, llamado, llamados, llamamos, llaman ...  (36 formas)
  aloj       <- aloja, alojaba, alojaban, alojada, alojadas, alojado, alojados, alojamiento, alojamientos, alojamos ...  (33 formas)
  qued       <- queda, quedaba, quedaban, quedabas, quedado, quedamos, quedan, quedando, quedar, quedara ...  (33 formas)
  llev       <- lleva, llevaba, llevaban, llevabas, llevada, llevado, llevamos, llevan, llevando, llevar ...  (32 formas)
  pag        <- pag, paga, pagaba, pagaban, pagada, pagadas, pagado, pagados, pagamos, pagan ...  (32 formas)
  habl       <- habla, hablaba, hablaban, hablado, hablados, hablamos, hablan, hablando

### Tu decisión (tramo 5)
1. **Stopwords:** ¿lista completa, ninguna o curada? Cita las palabras de 5.1 que justifican tu respuesta.
2. **Stemming o lematización:** compara la tabla de 5.3 (vocabulario, tiempo, memoria) con los ejemplos.
   ¿Qué pierdes con cada opción? ¿Cambia tu respuesta si el corpus fuera 100 veces mayor?
3. **Interacción con acentos:** el lematizador necesita los acentos para analizar bien el verbo.
   ¿Qué implica eso para la decisión de quitar acentos del tramo 3 y en qué orden aplicarías cada paso?
4. Mi decisión: **________**

## 6. Tokenización: palabra frente a subpalabra
**Regla de oro:** todo lo que se *aprende* de los datos (vocabulario, fusiones BPE, TF-IDF) se ajusta
**solo con entrenamiento**. Por eso la partición va primero y el resto de este notebook respeta la separación.

In [19]:
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_val_score
from sklearn.pipeline import Pipeline
from tokenizers import Tokenizer, models, pre_tokenizers, trainers

idx_train, idx_test = train_test_split(
    df.index.to_numpy(), test_size=0.2, stratify=df["label"], random_state=SEED
)
y_all = df["label"].to_numpy()
y_train, y_test = df.loc[idx_train, "label"].to_numpy(), df.loc[idx_test, "label"].to_numpy()

print(f"entrenamiento: {len(idx_train)}  |  prueba: {len(idx_test)}")
print("proporción de negativas (0): train {:.1%} / test {:.1%}".format((y_train == 0).mean(), (y_test == 0).mean()))
print("textos de prueba presentes en entrenamiento:",
      int(df.loc[idx_test, "review_text"].isin(set(df.loc[idx_train, "review_text"])).sum()))

entrenamiento: 2688  |  prueba: 673
proporción de negativas (0): train 61.1% / test 61.1%
textos de prueba presentes en entrenamiento: 0


### 6.1 Vocabulario por palabra y palabras fuera de vocabulario (OOV)

In [20]:
tok_train = clean.loc[idx_train].map(WORD.findall)
tok_test = clean.loc[idx_test].map(WORD.findall)

vocab_train = Counter(w for ts in tok_train for w in ts)
test_tokens = [w for ts in tok_test for w in ts]
test_types = set(test_tokens)

oov_token_rate = sum(w not in vocab_train for w in test_tokens) / len(test_tokens)
oov_type_rate = len(test_types - set(vocab_train)) / len(test_types)
hapax = sum(1 for n in vocab_train.values() if n == 1)
min2 = sum(1 for n in vocab_train.values() if n >= 2)

print(f"vocabulario de entrenamiento (palabras):        {len(vocab_train):,}")
print(f"  que aparecen una sola vez (hápax):            {hapax:,} ({hapax / len(vocab_train):.1%})")
print(f"  que aparecen 2 o más veces:                   {min2:,}")
print(f"tokens de prueba fuera de vocabulario (OOV):    {oov_token_rate:.2%}")
print(f"tipos de prueba fuera de vocabulario:           {oov_type_rate:.2%}")

vocabulario de entrenamiento (palabras):        16,018
  que aparecen una sola vez (hápax):            8,426 (52.6%)
  que aparecen 2 o más veces:                   7,592
tokens de prueba fuera de vocabulario (OOV):    3.68%
tipos de prueba fuera de vocabulario:           28.22%


### 6.2 BPE entrenado solo con entrenamiento
Entrenamos un BPE propio (Hugging Face `tokenizers`) con varios tamaños de vocabulario y medimos:
tokens por reseña, tokens por palabra (*fertilidad*) y `[UNK]` en prueba.
No usamos un modelo preentrenado para mantener la tarea ligera y sin descargas adicionales.

In [21]:
def train_bpe(corpus: list[str], vocab_size: int) -> Tokenizer:
    """Entrena un tokenizador BPE.

    Args:
        corpus: Textos de entrenamiento.
        vocab_size: Tamaño objetivo del vocabulario.

    Returns:
        Tokenizador entrenado, con ``[UNK]`` para símbolos no vistos.
    """
    tk = Tokenizer(models.BPE(unk_token="[UNK]"))
    tk.pre_tokenizer = pre_tokenizers.Whitespace()
    trainer = trainers.BpeTrainer(vocab_size=vocab_size, special_tokens=["[UNK]"], show_progress=False)
    tk.train_from_iterator(corpus, trainer=trainer)
    return tk


train_corpus = clean.loc[idx_train].tolist()
test_corpus = clean.loc[idx_test].tolist()
ws =pre_tokenizers.Whitespace()
n_test_words = sum(len(ws.pre_tokenize_str(s)) for s in test_corpus)

rows = [
    {
        "tokenización": "palabra (regex \\w+)",
        "vocabulario": len(vocab_train),
        "tokens_por_reseña": round(len(test_tokens) / len(test_corpus), 1),
        "tokens_por_palabra": 1.0,
        "unk_o_oov_%": round(oov_token_rate * 100, 2),
        "entrenar_s": 0.0,
    }
]
bpes: dict[int, Tokenizer] = {}
for size in (1000, 2000, 4000, 8000):
    t0 = time.perf_counter()
    tk = train_bpe(train_corpus, size)
    secs = time.perf_counter() - t0
    bpes[size] = tk
    enc = [tk.encode(s).tokens for s in test_corpus]
    n_tok = sum(len(e) for e in enc)
    n_unk = sum(t == "[UNK]" for e in enc for t in e)
    rows.append(
        {
            "tokenización": f"BPE-{size}",
            "vocabulario": tk.get_vocab_size(),
            "tokens_por_reseña": round(n_tok / len(test_corpus), 1),
            "tokens_por_palabra": round(n_tok / n_test_words, 2),
            "unk_o_oov_%": round(100 * n_unk / n_tok, 3),
            "entrenar_s": round(secs, 2),
        }
    )
display(pd.DataFrame(rows))

,tokenización,vocabulario,tokens_por_reseña,tokens_por_palabra,unk_o_oov_%,entrenar_s
0,palabra (regex \w+),16018,94.9,1.00,3.680,0.00
1,BPE-1000,1000,158.1,1.48,0.032,0.14
2,BPE-2000,2000,136.9,1.28,0.037,0.16
3,BPE-4000,4000,124.1,1.16,0.041,0.22
4,BPE-8000,8000,116.4,1.09,0.043,0.31


In [22]:
rng = random.Random(SEED)
oov_words = sorted(test_types - set(vocab_train))
shown = rng.sample(oov_words, 10)
print("palabras de prueba que el vocabulario de palabras NO conoce, y cómo las parte el BPE-4000:")
for w in shown:
    print(f"  {w:<24} -> {bpes[4000].encode(w).tokens}")

print("\npalabras frecuentes bien escritas frente a su versión con error tipográfico:")
for w in ["habitaciones", "recepcionista", "horribles", "habtiaciones", "habitaciòn", "maravillosoo"]:
    print(f"  {w:<24} -> {bpes[4000].encode(w).tokens}")

palabras de prueba que el vocabulario de palabras NO conoce, y cómo las parte el BPE-4000:
  concentran               -> ['con', 'cen', 'tran']
  alteración               -> ['al', 'tera', 'ción']
  irreprochable            -> ['ir', 're', 'pro', 'chab', 'le']
  hallowen                 -> ['h', 'al', 'lo', 'w', 'en']
  filigranas               -> ['fi', 'li', 'gran', 'as']
  defrauda                 -> ['de', 'frau', 'da']
  colocados                -> ['colo', 'cados']
  ceno                     -> ['cen', 'o']
  remedios                 -> ['re', 'medi', 'os']
  antideslizante           -> ['anti', 'des', 'li', 'z', 'ante']

palabras frecuentes bien escritas frente a su versión con error tipográfico:
  habitaciones             -> ['habitaciones']
  recepcionista            -> ['recepcionista']
  horribles                -> ['horribles']
  habtiaciones             -> ['hab', 'ti', 'a', 'ciones']
  habitaciòn               -> ['habita', 'ci', 'ò', 'n']
  maravillosoo             -> ['

### Tu decisión (tramo 6)
1. **Palabra o subpalabra:** ¿cuál usas para el baseline y por qué? Apóyate en OOV, fertilidad y los ejemplos de arriba.
2. ¿Qué ventaja da la subpalabra con errores tipográficos? ¿Qué costo tiene en interpretabilidad?
3. ¿Qué tamaño de vocabulario elegirías para BPE y qué observas al subir de 1000 a 8000?
4. Mi decisión: **________**

## 7. Baseline TF-IDF, eficiencia y sesgos
**Protocolo** (para no engañarnos con 3,361 reseñas):
- Las variantes de preprocesamiento se comparan con **validación cruzada de 5 pliegues solo sobre entrenamiento**.
- El conjunto de prueba se usa **una sola vez**, con el baseline mínimo y con tu configuración.
  Si eligiéramos la mejor variante mirando prueba, el resultado quedaría contaminado.
- Métrica principal: **macro-F1**, porque las clases no están balanceadas (61% / 39%).

### 7.1 Variantes de preprocesamiento

In [23]:
CURATED_NOACC = {strip_accents(w) for w in CURATED}


def join_tokens(tokens: list[str]) -> str:
    """Une tokens con espacios.

    Args:
        tokens: Lista de tokens.

    Returns:
        Cadena con tokens separados por un espacio.
    """
    return " ".join(tokens)


noacc_clean = df["review_text"].map(lambda s: clean_text(s, strip_acc=True))

VARIANTS: dict[str, tuple[pd.Series, str]] = {
    "A mínimo (solo minúsculas)": (df["review_text"].str.lower(), "word"),
    "B limpieza propuesta": (clean, "word"),
    "C B + stopwords lista completa": (tok.map(lambda ts: join_tokens(drop_words(ts, STOP_WORDS))), "word"),
    "D B + stopwords curada": (tok.map(lambda ts: join_tokens(drop_words(ts, CURATED))), "word"),
    "E D sin acentos": (noacc_clean.map(lambda s: join_tokens(drop_words(WORD.findall(s), CURATED_NOACC))), "word"),
    "F D + lematización": (tok.map(lambda ts: join_tokens([lemma_of.get(w, w) for w in drop_words(ts, CURATED)])), "word"),
    "G D + stemming": (tok.map(lambda ts: join_tokens([stem_of.get(w, w) for w in drop_words(ts, CURATED)])), "word"),
    "H B + BPE-4000": (clean.map(lambda s: join_tokens(bpes[4000].encode(s).tokens)), "split"),
}
for name, (series, _) in VARIANTS.items():
    print(f"{name:<34} ejemplo: {series.iloc[3][:90]!r}")

A mínimo (solo minúsculas)         ejemplo: 'son pequeños, se bloquean y son insuficientes para el tamaño del hotel. la mayoría de los '
B limpieza propuesta               ejemplo: 'son pequeños, se bloquean y son insuficientes para el tamaño del hotel. la mayoría de los '
C B + stopwords lista completa     ejemplo: 'pequeños bloquean insuficientes tamaño hotel mayoría clientes tenían escaleras estuve num '
D B + stopwords curada             ejemplo: 'pequeños bloquean insuficientes tamaño hotel mayoría clientes tenían escaleras estuve num '
E D sin acentos                    ejemplo: 'pequenos bloquean insuficientes tamano hotel mayoria clientes tenian escaleras estuve num '
F D + lematización                 ejemplo: 'pequeño bloquear insuficiente tamaño hotel mayoría cliente tener escalera estar num planta'
G D + stemming                     ejemplo: 'pequeñ bloqu insuficient tamañ hotel mayor client ten escaler estuv num plant llegadan fun'
H B + BPE-4000                     ejempl

### 7.2 Comparación por validación cruzada (solo entrenamiento)

In [24]:
def make_pipeline(kind: str) -> Pipeline:
    """Construye TF-IDF + regresión logística.

    Args:
        kind: ``word`` para tokens por regex; ``split`` para texto ya separado por espacios.

    Returns:
        Pipeline de scikit-learn sin ajustar.
    """
    if kind == "word":
        vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True,
                              token_pattern=r"(?u)\b\w+\b", lowercase=False)
    else:
        vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True,
                              tokenizer=str.split, token_pattern=None, lowercase=False)
    clf = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)
    return Pipeline([("tfidf", vec), ("clf", clf)])


HPARAMS = {
    "tfidf": {"ngram_range": [1, 2], "min_df": 2, "sublinear_tf": True},
    "logreg": {"max_iter": 1000, "class_weight": "balanced", "C": 1.0},
    "cv": {"folds": 5, "shuffle": True},
}
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

dummy_cv = cross_val_score(DummyClassifier(strategy="most_frequent"),
                           clean.loc[idx_train], y_train, cv=skf, scoring="f1_macro")
rows = [{"variante": "azar: siempre la clase mayoritaria", "f1_macro_cv": dummy_cv.mean().round(3),
         "desv": dummy_cv.std().round(3), "vocab_tfidf": 0}]

for name, (series, kind) in VARIANTS.items():
    X_tr = series.loc[idx_train]
    scores = cross_val_score(make_pipeline(kind), X_tr, y_train, cv=skf, scoring="f1_macro")
    n_feat = len(make_pipeline(kind).fit(X_tr, y_train).named_steps["tfidf"].vocabulary_)
    rows.append({"variante": name, "f1_macro_cv": scores.mean().round(3),
                 "desv": scores.std().round(3), "vocab_tfidf": n_feat})

cv_table = pd.DataFrame(rows)
base = float(cv_table.loc[cv_table["variante"].str.startswith("A "), "f1_macro_cv"].iloc[0])
cv_table["Δ vs A"] = (cv_table["f1_macro_cv"] - base).round(3)
display(cv_table)
print("\nRegla práctica: una diferencia menor que la desviación entre pliegues (columna desv) no es evidencia suficiente.")

,variante,f1_macro_cv,desv,vocab_tfidf,Δ vs A
0,azar: siempre la clase mayoritaria,0.379,0.000,0,-0.585
1,A mínimo (solo minúsculas),0.964,0.006,31386,0.000
2,B limpieza propuesta,0.964,0.006,31012,0.000
3,C B + stopwords lista completa,0.956,0.009,14327,-0.008
4,D B + stopwords curada,0.962,0.008,16984,-0.002
5,E D sin acentos,0.962,0.007,16826,-0.002
6,F D + lematización,0.964,0.009,16716,0.000
7,G D + stemming,0.966,0.008,16403,0.002
8,H B + BPE-4000,0.965,0.006,40891,0.001



Regla práctica: una diferencia menor que la desviación entre pliegues (columna desv) no es evidencia suficiente.


### 7.3 Evaluación final (una sola vez) y eficiencia
Se evalúa en prueba el **baseline mínimo (A)** y tu **configuración propuesta (D: limpieza + stopwords curada)**.

In [25]:
import tracemalloc
from datetime import datetime

Path("experiments.jsonl").write_text("", encoding="utf-8")


def log_experiment(name: str, params: dict, metrics: dict) -> None:
    """Agrega un registro de experimento a ``experiments.jsonl``.

    Args:
        name: Nombre del experimento.
        params: Hiperparámetros y decisiones de preprocesamiento.
        metrics: Métricas obtenidas.
    """
    record = {
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "name": name,
        "seed": SEED,
        "params": params,
        "metrics": metrics,
        "data_sha256": run_meta["data_sha256"],
        "commit": run_meta["commit"],
    }
    with Path("experiments.jsonl").open("a", encoding="utf-8") as fh:
        fh.write(json.dumps(record, ensure_ascii=False) + "\n")


final_rows = []
fitted: dict[str, Pipeline] = {}
for name in ["A mínimo (solo minúsculas)", "D B + stopwords curada"]:
    series, kind = VARIANTS[name]
    pipe = make_pipeline(kind)

    tracemalloc.start()
    t0 = time.perf_counter()
    pipe.fit(series.loc[idx_train], y_train)
    fit_s = time.perf_counter() - t0
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    t0 = time.perf_counter()
    pred = pipe.predict(series.loc[idx_test])
    pred_s = time.perf_counter() - t0

    X = pipe.named_steps["tfidf"].transform(series.loc[idx_train])
    size_mb = (X.data.nbytes + X.indices.nbytes + X.indptr.nbytes) / 1e6
    metrics = {
        "accuracy": round(accuracy_score(y_test, pred), 4),
        "f1_macro": round(f1_score(y_test, pred, average="macro"), 4),
        "fit_s": round(fit_s, 2),
        "predict_s": round(pred_s, 3),
        "peak_python_mb": round(peak / 1e6, 1),
        "matriz_tfidf_mb": round(size_mb, 2),
        "n_features": X.shape[1],
    }
    final_rows.append({"variante": name, **metrics})
    fitted[name] = pipe
    log_experiment(name, {"variant": name, **HPARAMS}, metrics)

display(pd.DataFrame(final_rows))

t0 = time.perf_counter()
_ = texts.map(clean_text)
print(f"\nlimpieza de {len(texts)} reseñas: {time.perf_counter() - t0:.2f} s")
print("\ninforme por clase (configuración D, prueba):")
print(classification_report(y_test, fitted["D B + stopwords curada"].predict(VARIANTS["D B + stopwords curada"][0].loc[idx_test]),
                            target_names=["negativa (0)", "positiva (1)"], digits=3))

,variante,accuracy,f1_macro,fit_s,predict_s,peak_python_mb,matriz_tfidf_mb,n_features
0,A mínimo (solo minúsculas),0.9510,0.9485,2.44,0.099,33.9,3.88,31386
1,D B + stopwords curada,0.9539,0.9517,1.40,0.054,28.5,1.63,16984



limpieza de 3361 reseñas: 1.91 s

informe por clase (configuración D, prueba):
              precision    recall  f1-score   support

negativa (0)      0.966     0.959     0.962       411
positiva (1)      0.936     0.947     0.941       262

    accuracy                          0.954       673
   macro avg      0.951     0.953     0.952       673
weighted avg      0.954     0.954     0.954       673



### 7.4 Revisión crítica: ¿en qué se fija el modelo?
Un clasificador puede acertar por razones equivocadas (nombres de hoteles, ciudades, personas).
Miramos los términos con mayor peso en cada clase.

In [26]:
pipe_d = fitted["D B + stopwords curada"]
names = np.array(pipe_d.named_steps["tfidf"].get_feature_names_out())
coef = pipe_d.named_steps["clf"].coef_[0]  # positivo = clase 1 (positiva)
order = np.argsort(coef)
print("términos que más empujan a NEGATIVA:")
print("  ", ", ".join(names[order[:20]]))
print("\ntérminos que más empujan a POSITIVA:")
print("  ", ", ".join(names[order[::-1][:20]]))

términos que más empujan a NEGATIVA:
   no, ni, mal, pero, peor, nada, mala, num_precio, poco, noche, dormir, num_estrellas, nadie, no recomiendo, ruido, reserva, paredes, viejo, fotos, sucio

términos que más empujan a POSITIVA:
   sevilla, muy, excelente, personal, muy bien, gracias, duda, catedral, centro, ubicación, limpio, terraza, atención, perfecto, sin duda, cerca, recomendable, desayuno, volveremos, agradable


### 7.5 Sesgos y desempeño por subgrupo
Usamos predicciones **fuera de pliegue** (validación cruzada sobre las 3,361 reseñas) para tener más casos por grupo.
Los grupos con menos de 100 reseñas se marcan como poco fiables.

**No eliminamos identidades ni dialectos**: los medimos. Los grupos léxicos de identidad son una aproximación
por palabras clave, no una etiqueta real de quién escribió la reseña.

In [27]:
series_d, kind_d = VARIANTS["D B + stopwords curada"]
oof = cross_val_predict(make_pipeline(kind_d), series_d, y_all, cv=skf)

raw_text = df["review_text"]
city = df["location"].str.split("_").str[0]
top_cities = city.value_counts().head(4).index.tolist()
length_q = pd.qcut(raw_text.str.len(), 3, labels=["corto", "medio", "largo"])

UNACCENTED = r"\b(habitacion(es)?|recepcion|ubicacion|atencion|situacion|habia|ademas|tambien|aqui|asi)\b"
IDENTITY = {
    "familia con niños": r"\b(niñ[oa]s?|hij[oa]s?|bebés?|bebe)\b",
    "pareja / matrimonio": r"\b(pareja|marido|mujer|esposa|esposo|novi[oa])\b",
    "mención de extranjeros o nacionalidad": r"\b(extranjer\w+|ingles\w*|alemán\w*|alemanes|american\w+|frances\w*)\b",
    "personas mayores o movilidad reducida": r"\b(mayores|anciano\w*|jubilad\w+|abuel\w+|discapacitad\w+|silla de ruedas)\b",
}


def group_report(mask: pd.Series, name: str) -> dict:
    """Métricas de la predicción fuera de pliegue para un subgrupo.

    Args:
        mask: Máscara booleana alineada con ``df``.
        name: Nombre del grupo.

    Returns:
        Diccionario con tamaño, proporción de negativas, exactitud y macro-F1.
    """
    sel = mask.to_numpy()
    yt, yp = y_all[sel], oof[sel]
    both = len(set(yt)) > 1
    return {
        "grupo": name,
        "n": int(sel.sum()),
        "pct_negativas": round(float((yt == 0).mean() * 100), 1) if sel.any() else np.nan,
        "accuracy": round(float(accuracy_score(yt, yp)), 3) if sel.any() else np.nan,
        "f1_macro": round(float(f1_score(yt, yp, average="macro")), 3) if both else np.nan,
        "fiable_n>=100": bool(sel.sum() >= 100),
    }


groups = [group_report(pd.Series(True, index=df.index), "TODAS")]
groups += [group_report(df["hotel"].isna(), "fuente probable: corpus COAH 2014 (hotel vacío)"),
           group_report(df["hotel"].notna(), "fuente probable: TripAdvisor 2021 (hotel presente)")]
groups += [group_report(city == c, f"ciudad: {c}") for c in top_cities]
groups += [group_report(length_q == q, f"largo: {q}") for q in ["corto", "medio", "largo"]]
groups += [group_report(raw_text.str.contains(UNACCENTED, case=False, regex=True), "ortografía informal (sin acentos habituales)"),
           group_report(~raw_text.str.contains(UNACCENTED, case=False, regex=True), "sin esa marca informal"),
           group_report(has_emoji, "con emoji")]
groups += [group_report(raw_text.str.contains(rx, case=False, regex=True), f"identidad: {name}") for name, rx in IDENTITY.items()]
display(pd.DataFrame(groups))

C:\Users\juans\AppData\Local\Temp\ipykernel_32664\1024953914.py:46: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  groups += [group_report(raw_text.str.contains(UNACCENTED, case=False, regex=True), "ortografía informal (sin acentos habituales)"),
C:\Users\juans\AppData\Local\Temp\ipykernel_32664\1024953914.py:47: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  group_report(~raw_text.str.contains(UNACCENTED, case=False, regex=True), "sin esa marca informal"),
C:\Users\juans\AppData\Local\Temp\ipykernel_32664\1024953914.py:49: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  groups += [group_report(raw_text.str.contains(rx, case=False, regex=True), f"identidad: {name}") for name, rx in IDENTITY.items()]


C:\Users\juans\AppData\Local\Temp\ipykernel_32664\1024953914.py:49: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  groups += [group_report(raw_text.str.contains(rx, case=False, regex=True), f"identidad: {name}") for name, rx in IDENTITY.items()]
C:\Users\juans\AppData\Local\Temp\ipykernel_32664\1024953914.py:49: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  groups += [group_report(raw_text.str.contains(rx, case=False, regex=True), f"identidad: {name}") for name, rx in IDENTITY.items()]
C:\Users\juans\AppData\Local\Temp\ipykernel_32664\1024953914.py:49: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  groups += [group_report(raw_text.str.contains(rx, case=False, regex=True), f"identidad: {name}") for name, rx in IDENTITY.items()

,grupo,n,pct_negativas,accuracy,f1_macro,fiable_n>=100
0,TODAS,3361,61.1,0.963,0.961,True
1,fuente probable: corpus COAH 2014 (hotel vacío),500,100.0,0.988,NaN,True
2,fuente probable: TripAdvisor 2021 (hotel prese...,2861,54.3,0.958,0.958,True
3,ciudad: Seville,1749,25.2,0.947,0.933,True
4,ciudad: Granada,266,100.0,0.959,NaN,True
5,ciudad: Cordoba,266,100.0,0.981,NaN,True
6,ciudad: Malaga,251,100.0,0.972,NaN,True
7,largo: corto,1127,35.4,0.959,0.955,True
8,largo: medio,1114,62.3,0.960,0.957,True
9,largo: largo,1120,85.7,0.970,0.936,True


### Tu decisión (tramo 7)
1. **Qué variante elegirías** según la tabla 7.2 y por qué. ¿Hay diferencias mayores que la desviación entre pliegues?
2. **Eficiencia:** ¿por qué TF-IDF es una decisión razonable antes de escalar a un modelo neuronal? Usa tiempo, memoria y tamaño de la matriz.
3. **Sesgos:** ¿qué grupos muestran peor desempeño y cuánto confías en esa conclusión dado su tamaño?
4. **Mitigación sin borrar identidades ni dialectos:** ¿qué harías en vez de eliminar esas señales?
5. Mi decisión: **________**